# Quantum Computing RAG Pipeline Evaluation with LangSmith & LLM as Judge

This notebook evaluates our modular FAISS + Groq RAG pipeline using LangSmith and LLM as judge.
It follows the exact structure and metrics from the reference evaluation framework:

1. **Answer Correctness**: Assesses factual accuracy against ground truth answers.
2. **Answer Relevance**: Checks whether the response directly addresses the user question.
3. **Groundedness / Faithfulness**: Verifies zero hallucinations beyond retrieved context documents.
4. **Retrieval Relevance**: Ensures retrieved document chunks are relevant to the query.

## 1. Setup & Environment Variables

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

# Configure LangSmith credentials & EU endpoint
langsmith_key = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGSMith_API_KEY")
if langsmith_key:
    os.environ["LANGSMITH_API_KEY"] = langsmith_key

langsmith_endpoint = os.getenv("LANGSMITH_ENDPOINT", "https://eu.api.smith.langchain.com")
os.environ["LANGSMITH_ENDPOINT"] = langsmith_endpoint
os.environ["LANGSMITH_TRACING"] = "true"

# Ensure src directory is accessible
if "src" not in sys.path:
    sys.path.append(os.path.abspath("src"))

from langsmith import Client
client = Client(api_url=langsmith_endpoint, api_key=langsmith_key)
print(f"Connected to LangSmith endpoint: {langsmith_endpoint}")

## 2. Test Dataset (Quantum Computing Benchmark)

In [ ]:
dataset_name = "Quantum RAG Evaluation"

examples = [
    {
        "inputs": {"question": "What is a qubit in quantum computing?"},
        "outputs": {
            "answer": "A qubit is the fundamental unit of quantum information, analogous to a classical bit, which can exist in a superposition of states |0> and |1> in a two-dimensional Hilbert space."
        }
    },
    {
        "inputs": {"question": "What is the no-cloning theorem?"},
        "outputs": {
            "answer": "The no-cloning theorem states that it is impossible to create an identical copy of an arbitrary unknown quantum state."
        }
    },
    {
        "inputs": {"question": "How does an n-qubit quantum system scale in Hilbert space size?"},
        "outputs": {
            "answer": "An n-qubit quantum system has access to a Hilbert space of dimension 2^n, enabling exponential state representations compared to classical bits."
        }
    },
    {
        "inputs": {"question": "What are Variational Quantum Eigensolvers used for?"},
        "outputs": {
            "answer": "Variational Quantum Eigensolvers (VQE) are hybrid quantum-classical algorithms that use parameterized quantum circuits and classical optimization to find the ground state energy of quantum systems."
        }
    },
    {
        "inputs": {"question": "What is quantum decoherence?"},
        "outputs": {
            "answer": "Quantum decoherence is the decay of quantum superposition into classical states caused by environmental interaction and noise."
        }
    }
]

use_langsmith = False
try:
    existing_datasets = list(client.list_datasets(dataset_name=dataset_name))
    if existing_datasets:
        dataset = existing_datasets[0]
        print(f"Found existing LangSmith cloud dataset: {dataset.name} (id: {dataset.id})")
    else:
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description="Quantum Computing benchmark evaluation dataset."
        )
        client.create_examples(dataset_id=dataset.id, examples=examples)
        print(f"Created LangSmith cloud dataset '{dataset_name}' with {len(examples)} examples.")
    use_langsmith = True
except Exception as e:
    print(f"Notice: LangSmith cloud dataset sync skipped ({e}). Continuing with local benchmark dataset.")
    dataset = examples

## 3. Define Metric Schemas & Instructions

In [ ]:
from typing import Any
from typing_extensions import Annotated, TypedDict
from langchain_groq import ChatGroq

class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

def get_grader_llm(schema: Any):
    groq_key = os.getenv("GROQ_API_KEY")
    eval_model = os.getenv("GROQ_EVAL_MODEL", "openai/gpt-oss-20b")
    return ChatGroq(groq_api_key=groq_key, model_name=eval_model, temperature=0).with_structured_output(schema)

print("Metric schemas and instructions initialized.")

## 4. Evaluator Functions with `_get_val` Extraction

In [ ]:
def _get_val(obj: Any, key: str) -> bool:
    if isinstance(obj, dict):
        return bool(obj.get(key, False))
    return bool(getattr(obj, key, False))

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    answers = f"QUESTION: {inputs['question']}\nGROUND TRUTH ANSWER: {reference_outputs['answer']}\nSTUDENT ANSWER: {outputs['answer']}"
    grader = get_grader_llm(CorrectnessGrade)
    grade = grader.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers},
    ])
    return _get_val(grade, "correct")

def relevance(inputs: dict, outputs: dict) -> bool:
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grader = get_grader_llm(RelevanceGrade)
    grade = grader.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return _get_val(grade, "relevant")

def groundedness(inputs: dict, outputs: dict) -> bool:
    docs = outputs.get("documents", [])
    doc_string = "\n\n".join(doc.page_content if hasattr(doc, "page_content") else str(doc) for doc in docs)
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grader = get_grader_llm(GroundedGrade)
    grade = grader.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer},
    ])
    return _get_val(grade, "grounded")

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    docs = outputs.get("documents", [])
    doc_string = "\n\n".join(doc.page_content if hasattr(doc, "page_content") else str(doc) for doc in docs)
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    grader = get_grader_llm(RetrievalRelevanceGrade)
    grade = grader.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return _get_val(grade, "relevant")

print("All 4 evaluators registered successfully.")

## 5. RAG Pipeline Target Adapter

In [ ]:
from langsmith import traceable
from src.search import RAGSearch

rag_search = RAGSearch(persist_dir="src/faiss_store", llm_model="openai/gpt-oss-20b")

@traceable(name="quantum_rag_pipeline")
def rag_bot(question: str) -> dict:
    return rag_search.search_and_retrieve(question, top_k=3, score_threshold=0.45)

def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

sample = target({"question": "What is a qubit in quantum computing?"})
print("Sample Answer:", sample["answer"][:120], "...")
print("Retrieved Documents Count:", len(sample.get("documents", [])))

## 6. Run Evaluation & View Results

In [ ]:
import pandas as pd
from src.evaluation import print_evaluation_summary

df = None
if use_langsmith:
    try:
        print("Running cloud evaluation on LangSmith...")
        experiment_results = client.evaluate(
            target,
            data=dataset_name,
            evaluators=[correctness, groundedness, relevance, retrieval_relevance],
            experiment_prefix="quantum-rag-eval",
            metadata={"pipeline": "FAISS-MiniLM-Groq", "top_k": 3}
        )
        df = experiment_results.to_pandas()
    except Exception as e:
        print(f"Notice: LangSmith cloud evaluate failed ({e}). Running local fallback...")

if df is None:
    from src.evaluation import run_evaluation_local
    df = run_evaluation_local(examples)
else:
    print_evaluation_summary(df)

df